# Phase 3 — Fine-tune a long-context checkpoint on few-shot batches

Workflow:

1. Load weights from a model trained with **long** `seq_in_len` (e.g. `phase1_improved` checkpoint). The first projection layers still see 13-D features per timestep; only the **temporal** extent changes, so transfer is reasonable.
2. Train a few epochs on `FewShotPaddedMotionDataset` with **small** `min_context` / `max_context` and **padding masks**, using a **lower learning rate** to avoid catastrophic forgetting.

Set `PRETRAINED_PATH` to an existing file under `pretrained/`, or skip loading to train from scratch on few-shot data only.

**Architecture match:** `MotionTransformer` hyperparameters here must match the checkpoint (e.g. `d_model`, `nhead`, `num_layers`, `dim_ff`); otherwise `strict=False` loads only overlapping weights and most layers stay random.

In [ ]:
import os
import torch
from torch import optim
from torch.utils.data import DataLoader

from few_shot_motion_dataset import FewShotPaddedMotionDataset, build_src_key_padding_mask
from transformer_encoder import MotionTransformer
from loss import LossFunction

PRETRAINED_PATH = "pretrained/transformer-encoder-d256-ff512-nh32-1l-n3.pth"
MAX_CONTEXT = 20
MIN_CONTEXT = 2
SEQ_OUT_LEN = 20
BASE_DIR = os.environ.get("MOT_DATASET_ROOT", "../../Datasets/")

train_ds = FewShotPaddedMotionDataset.from_roots(
    [f"{BASE_DIR}MOT17/train"],
    max_context=MAX_CONTEXT,
    min_context=MIN_CONTEXT,
    seq_out_len=SEQ_OUT_LEN,
    steps=4,
    noise_prob=0.12,
    noise_coeff=0.12,
    samples_per_window=2,
    seed=0,
    return_context_len=True,
)
val_ds = FewShotPaddedMotionDataset.from_roots(
    [f"{BASE_DIR}MOT17/val"],
    max_context=MAX_CONTEXT,
    min_context=MIN_CONTEXT,
    seq_out_len=SEQ_OUT_LEN,
    steps=4,
    noise_prob=0.12,
    noise_coeff=0.12,
    samples_per_window=1,
    seed=1,
    return_context_len=True,
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = MotionTransformer(
    input_dim=13,
    output_dim=5,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_ff=1024,
    dropout=0.1,
).to(DEVICE)

if os.path.isfile(PRETRAINED_PATH):
    state = torch.load(PRETRAINED_PATH, map_location=DEVICE, weights_only=True)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print("loaded", PRETRAINED_PATH, "missing", len(missing), "unexpected", len(unexpected))
else:
    print("no checkpoint at", PRETRAINED_PATH, "— training from init")

crit = LossFunction(loss1_coeff=1.0, loss2_coeff=0.35, loss3_coeff=0.25, loss4_coeff=0.0)
opt = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)


def train_epoch():
    model.train()
    t = 0.0
    for src, trg, _, gt_trg, k in train_loader:
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        gt_trg = gt_trg.to(DEVICE)
        pad = build_src_key_padding_mask(k, MAX_CONTEXT, DEVICE)
        opt.zero_grad()
        out = model(src, trg[:, :-1], src_key_padding_mask=pad)
        loss = crit(out, gt_trg[:, 1:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        t += loss.item()
    return t / max(len(train_loader), 1)


@torch.no_grad()
def val_epoch():
    model.eval()
    t = 0.0
    for src, trg, _, gt_trg, k in val_loader:
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        gt_trg = gt_trg.to(DEVICE)
        pad = build_src_key_padding_mask(k, MAX_CONTEXT, DEVICE)
        out = model(src, trg[:, :-1], src_key_padding_mask=pad)
        t += crit(out, gt_trg[:, 1:]).item()
    return t / max(len(val_loader), 1)


best = float("inf")
out_path = "pretrained/transformer_finetune_few_shot.pth"
os.makedirs("pretrained", exist_ok=True)
for ep in range(1, 11):
    tr = train_epoch()
    va = val_epoch()
    if va < best:
        best = va
        model.save_weight(out_path)
    print(f"finetune ep {ep:02d} train {tr:.5f} val {va:.5f}")
print("best", best, "->", out_path)